# 07b — E2 shortcut audit for M4 (Apratim Sharma)

The E2 test-time ablation of `07-e2-shortcut-audit`, applied to M4 (AST),
which was trained after E2 ran. Identical conditions (intact, low-pass at
7/6/5/4/3/2 kHz with the same zero-phase Butterworth, MP3 at 128/64/32 kbps
with the same libmp3lame round trip), identical 1,604 test clips, identical
summary statistics. The only difference is the front-end: the perturbed
waveform goes through `ASTFeatureExtractor`, as M4 was trained on.

**Prediction, written before the run.** M4 is pre-trained on AudioSet, a
corpus full of band-limited, compressed YouTube audio, so it has seen
low-passed and MP3-coded sound as normal input. If pre-training helps, M4
should degrade less than M2/M3 under MP3 in particular.

In [ ]:
import os, glob, json, time, shutil, subprocess, tempfile
import numpy as np, pandas as pd, torch
import transformers
from transformers import ASTFeatureExtractor, ASTForAudioClassification
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available()

def find_input(name, marker, required=True):
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/**/{marker}', recursive=True)
    if not hits:
        assert not required, f'{name}/{marker} not found'
        return None
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'subset.csv')
M4_DIR   = find_input('fmc-m4-results', 'm4_best_fp16.pt')
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
REAL_DIR = find_input('musiccaps-real-16k', '-0Gj8-vB1q4.wav')
OUT, SEED = '/kaggle/working', 42
CKPT_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

def find_ffmpeg():
    exe = shutil.which('ffmpeg')
    if exe and 'libmp3lame' in subprocess.run([exe, '-hide_banner', '-encoders'],
                                              capture_output=True, text=True).stdout:
        return exe
    subprocess.run(['pip', 'install', '-q', 'imageio-ffmpeg'], check=True)
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()
FFMPEG = find_ffmpeg()

sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
labels = np.load(f'{MELS_DIR}/labels.npy')
sub['path'] = (sub.path
    .str.replace('/kaggle/working/real', REAL_DIR, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', RAW_DIR, regex=False))
te = np.where((sub.split == 'test').values)[0]
y_te, gen_te = labels[te], sub.generator.values[te]
assert len(te) == 1604 and all(map(os.path.exists, sub.path.values[te]))

In [ ]:
import soundfile as sf
from scipy.signal import butter, sosfiltfilt, correlate
from joblib import Parallel, delayed

SR, N_SAMPLES = 16000, 160000

def fix_len(y):
    return y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))

def load(path):
    y, sr = sf.read(path, dtype='float32')
    assert sr == SR
    return fix_len(y)
def lowpass(y, fc):
    # 8th-order Butterworth run forwards and backwards (sosfiltfilt): zero
    # phase, so the waveform is not shifted in time, and a steep enough
    # roll-off that the band above fc is genuinely removed.
    sos = butter(8, fc, btype='low', fs=SR, output='sos')
    return sosfiltfilt(sos, y).astype(np.float32)

def mp3(y, kbps):
    # Real files rather than pipes: with a seekable output libmp3lame writes
    # the LAME header, and ffmpeg's decoder uses it to drop the encoder delay,
    # so the decoded clip lines up with the original sample for sample.
    with tempfile.TemporaryDirectory() as d:
        sf.write(f'{d}/in.wav', y, SR, subtype='FLOAT')
        subprocess.run([FFMPEG, '-y', '-v', 'error', '-i', f'{d}/in.wav',
                        '-c:a', 'libmp3lame', '-b:a', f'{kbps}k', f'{d}/x.mp3'], check=True)
        subprocess.run([FFMPEG, '-y', '-v', 'error', '-i', f'{d}/x.mp3',
                        '-ac', '1', '-ar', str(SR), '-c:a', 'pcm_f32le', f'{d}/out.wav'], check=True)
        z, sr = sf.read(f'{d}/out.wav', dtype='float32')
    assert sr == SR
    return fix_len(z)

CONDITIONS = [('intact', None, None)] + \
             [(f'lowpass_{fc // 1000}k', 'lowpass', fc) for fc in [7000, 6000, 5000, 4000, 3000, 2000]] + \
             [(f'mp3_{k}k', 'mp3', k) for k in [128, 64, 32]]

def perturb(y, kind, arg):
    if kind is None:      return y
    if kind == 'lowpass': return lowpass(y, arg)
    if kind == 'mp3':     return mp3(y, arg)
    raise ValueError(kind)


fe = ASTFeatureExtractor.from_pretrained(CKPT_NAME)

def process(path, kind, arg):
    y = perturb(load(path), kind, arg)
    return fe(y, sampling_rate=SR, return_tensors='np')['input_values'][0].astype(np.float16)

In [ ]:
# M4 from its saved fp16 checkpoint (cast back to fp32 weights; inference
# still runs under fp16 autocast exactly as in 09-m4-ast).
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
m4 = ASTForAudioClassification.from_pretrained(CKPT_NAME, num_labels=1, ignore_mismatched_sizes=True,
                                               attn_implementation='sdpa')
sd = torch.load(f'{M4_DIR}/m4_best_fp16.pt', map_location='cpu')
m4.load_state_dict({k: v.float() for k, v in sd.items()}); m4 = m4.cuda().eval()

def score_m4(FB):
    out = []
    with torch.no_grad():
        for i in range(0, len(FB), 32):
            x = torch.from_numpy(FB[i:i+32].astype(np.float32)).cuda()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out.append(torch.sigmoid(m4(x).logits.squeeze(1).float()).cpu().numpy())
    return np.concatenate(out)

def metrics(true, score):
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred, zero_division=0)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

def summarise(true, score, gens):
    m = {k: round(v, 4) for k, v in metrics(true, score).items()}
    pred = (score > 0.5).astype(int)
    m["specificity_real"] = round(float((pred[true == 0] == 0).mean()), 4)
    m["recall_per_generator"] = {g: round(float(pred[gens == g].mean()), 4)
                                 for g in sorted(set(gens[true == 1]))}
    m["mean_score_real"]  = round(float(score[true == 0].mean()), 4)
    m["mean_score_synth"] = round(float(score[true == 1].mean()), 4)
    return m

published = json.load(open(f'{M4_DIR}/m4_results.json'))['balanced_accuracy']

In [ ]:
RES = f'{OUT}/e2_m4_results.json'
results = {}
header = {"experiment": "E2_shortcut_audit_test_time", "model": "M4", "seed": SEED, "split": "test",
          "n_test": int(len(te)), "checkpoint": "fmc-m4-results/m4_best_fp16.pt",
          "published_bal_acc": published, "ffmpeg": FFMPEG,
          "lowpass_filter": "Butterworth order 8, sosfiltfilt (zero phase)",
          "mp3_codec": "libmp3lame CBR, decoded by ffmpeg to 16 kHz float",
          "front_end": "ASTFeatureExtractor on the perturbed waveform"}
for name, kind, arg in CONDITIONS:
    t0 = time.time()
    FB = np.stack(Parallel(n_jobs=4)(delayed(process)(p, kind, arg) for p in sub.path.values[te]))
    s = score_m4(FB)
    entry = {"kind": kind, "arg": arg, "M4": summarise(y_te, s, gen_te)}
    if kind is None:
        # Must reproduce the M4 test result from 09-m4-ast.
        print('intact bal acc', entry['M4']['balanced_accuracy'], '| published', published)
        assert abs(entry['M4']['balanced_accuracy'] - published) < 3e-3
    results[name] = entry
    json.dump({**header, "conditions": results}, open(RES, 'w'), indent=2)
    e = entry['M4']
    print(f'{name:12s} M4 bal {e["balanced_accuracy"]:.4f} spec {e["specificity_real"]:.3f} '
          f'rec {e["recall"]:.3f} auc {e["auc"]:.3f}  ({time.time()-t0:.0f}s)', flush=True)
rows = [{"condition": n, "model": "M4", "bal_acc": r["M4"]["balanced_accuracy"],
         "delta_vs_intact": round(r["M4"]["balanced_accuracy"] - results['intact']["M4"]["balanced_accuracy"], 4),
         "specificity_real": r["M4"]["specificity_real"], "recall_synth": r["M4"]["recall"],
         "auc": r["M4"]["auc"], "eer": r["M4"]["eer"],
         **{f'rec_{g}': v for g, v in r["M4"]["recall_per_generator"].items()}} for n, r in results.items()]
pd.DataFrame(rows).to_csv(f'{OUT}/e2_m4_summary.csv', index=False)
print(pd.DataFrame(rows).to_string(index=False))